# Resumable LVIS Fruits & Vegetables YOLO Training in Google Colab

This notebook is designed for a **fresh 80-epoch baseline**, an **exact resume after a Colab interruption**, or a **new fine-tuning stage from completed weights**. It protects completed-epoch progress by copying `last.pt`, `best.pt`, experiment arguments, and metrics to Google Drive after each checkpoint save.

> **Important:** A free Colab GPU is not guaranteed. This notebook will preserve completed epochs, but no standard YOLO workflow can resume the exact minibatch inside an epoch that was interrupted. Before every resume, run the setup and dataset-preparation cells again so the dataset exists at exactly the same local path.

Run the cells **from top to bottom**. Edit only the configuration cell before starting training.

## 1. Mount persistent storage and install a reproducible package version

Google Drive is the durable system of record. The `/content` runtime disk is intentionally used for training speed and may disappear when Colab disconnects. The training library is pinned to the version used in the original notebook. Pandas is explicitly pinned to the version required by the Colab runtime so the installer cannot select an incompatible major release.

In [1]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

# Use Colab's required pandas version. Do not use an open-ended pandas>= constraint here: it can install pandas 3.x,
# which conflicts with the managed Colab and RAPIDS packages in this runtime.
!pip -q install --upgrade --upgrade-strategy only-if-needed "pandas==2.2.3" "ultralytics==8.3.32"

import pandas as pd
import ultralytics
assert pd.__version__ == '2.2.3', f'Expected pandas 2.2.3; found {pd.__version__}. Restart the runtime once, then rerun this cell.'
print('Pandas version:', pd.__version__)
print('Ultralytics version:', ultralytics.__version__)

Mounted at /content/drive
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 887.0/887.0 kB 57.2 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Pandas version: 2.2.3
Ultralytics version: 8.3.32


## 2. Verify the assigned GPU

A GPU is required. The notebook uses automatic batch sizing because free Colab can allocate different GPU models and memory amounts on different days.

In [2]:
import os
import subprocess
import torch

assert torch.cuda.is_available(), (
    'No CUDA GPU is attached. In Colab choose Runtime > Change runtime type > T4 GPU (or another GPU), then reconnect.'
)

print('PyTorch:', torch.__version__)
print('GPU:', torch.cuda.get_device_name(0))
print('GPU memory (GiB):', round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))
subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,driver_version', '--format=csv,noheader'], check=False)

PyTorch: 2.11.0+cu128
GPU: Tesla T4
GPU memory (GiB): 14.56


CompletedProcess(args=['nvidia-smi', '--query-gpu=name,memory.total,driver_version', '--format=csv,noheader'], returncode=0)

## 3. Configuration — edit this cell only

Use a new experiment name for each scientific comparison. The default `fresh` mode trains a 63-class baseline from pretrained `yolo11m.pt` for 80 epochs. `resume_interrupted` is only for a `last.pt` produced by this notebook after an unexpected interruption. `extend_completed` starts a **new fine-tuning stage** from a completed `best.pt` or `last.pt`; it is not an exact resume.

Your existing YOLO-format dataset folder is read directly from `/content/drive/MyDrive/LVIS_Fruits_And_Vegetables`. The notebook copies that immutable Drive folder to local storage before every run, allowing rapid writable caching and a stable path for checkpoint resume.

In [3]:
from pathlib import Path

# ---------- Persistent folders in Google Drive ----------
DRIVE_ROOT = Path('/content/drive/MyDrive/FoodDetection')
DRIVE_SOURCE = DRIVE_ROOT / 'source'
DRIVE_EXPERIMENTS = DRIVE_ROOT / 'experiments'
DRIVE_EXPORTS = DRIVE_ROOT / 'exports'
for folder in (DRIVE_SOURCE, DRIVE_EXPERIMENTS, DRIVE_EXPORTS):
    folder.mkdir(parents=True, exist_ok=True)

# ---------- Dataset source ----------
# User-provided YOLO dataset folder. It must contain images/, labels/, and a dataset YAML (possibly nested).
# It is copied locally before every run so label/image caches are writable and training is faster than reading every batch from Drive.
DATA_SOURCE = 'drive_folder'                 # already set for your existing Drive folder
DRIVE_DATA_ZIP = DRIVE_SOURCE / 'lvis_fruits_yolo.zip'  # unused unless you deliberately switch DATA_SOURCE to 'drive_zip'
DRIVE_DATA_FOLDER = Path('/content/drive/MyDrive/LVIS_Fruits_And_Vegetables')
DATA_YAML_HINT = None                        # None = auto-detect the single .yaml/.yml file; set a relative path only if multiple YAML files exist
LOCAL_DATA_ROOT = Path('/content/lvis_fruits_yolo')  # DO NOT change after starting a resumable run

# ---------- Experiment identity ----------
EXPERIMENT = 'lvis_fruits_yolo11m_80_v1'      # use a unique name; never overwrite a prior experiment
RUN_MODE = 'resume_interrupted'                            # choose: 'fresh', 'resume_interrupted', 'extend_completed'
FRESH_MODEL = 'yolo11m.pt'                    # recommended fast/accuracy baseline
TARGET_EPOCHS = 80                            # total target for fresh + later exact resume
FINETUNE_EPOCHS = 40                          # new stage length if RUN_MODE='extend_completed'

# Use only for resume_interrupted or extend_completed. Edit the experiment path/name as needed.
PREVIOUS_CHECKPOINT = Path('/content/drive/MyDrive/lvis_epoch45_recovery/lvis_fruits_yolo11m_80_v1/weights/last.pt')

# ---------- Controlled training settings ----------
IMGSZ = 640
BATCH = -1                                    # automatic: chooses safe GPU-memory utilization on the assigned Colab GPU
WORKERS = 2                                   # matches typical Colab CPU allocation; more workers are not automatically faster
CACHE = 'disk'                                # local runtime cache; never cache the entire dataset in Drive
DEVICE = 0
SEED = 0
PATIENCE = 20                                 # stop only after 20 no-improvement epochs
SAVE_PERIOD = 5                               # retain an extra immutable epoch checkpoint every five epochs

# Explicit values reproduce the optimizer selected by the original run's optimizer='auto'.
OPTIMIZER = 'AdamW'
LR0 = 0.000149
MOMENTUM = 0.9
WEIGHT_DECAY = 0.0005
WARMUP_EPOCHS = 3.0
LRF = 0.01

# Keep YOLO's validated detection-loss balance for the first experiment.
BOX_LOSS_GAIN = 7.5
CLS_LOSS_GAIN = 0.5
DFL_LOSS_GAIN = 1.5

# Evaluation/test options. Point TEST_DATA_YAML to an independently labelled test YAML only after it is uploaded.
TEST_DATA_YAML = None
PREDICT_SOURCE = None                         # e.g. DRIVE_SOURCE / 'test_images'
PREDICT_CONF = 0.25

LOCAL_RUNS_ROOT = Path('/content/yolo_runs')
LOCAL_EXPERIMENT = LOCAL_RUNS_ROOT / EXPERIMENT
DRIVE_EXPERIMENT = DRIVE_EXPERIMENTS / EXPERIMENT

print('Drive source:', DRIVE_SOURCE)
print('Drive experiment:', DRIVE_EXPERIMENT)
print('Mode:', RUN_MODE)
print('Fresh-model baseline:', FRESH_MODEL)

Drive source: /content/drive/MyDrive/FoodDetection/source
Drive experiment: /content/drive/MyDrive/FoodDetection/experiments/lvis_fruits_yolo11m_80_v1
Mode: resume_interrupted
Fresh-model baseline: yolo11m.pt


## 4. Create a local writable dataset copy and a stable local YAML

This avoids the original notebook's read-only cache warning. It also prevents a hidden mismatch between a downloaded dataset and a different hard-coded YAML path. This cell validates the dataset source before any GPU time is spent.

In [4]:
import shutil
import zipfile
import yaml

if LOCAL_DATA_ROOT.exists():
    shutil.rmtree(LOCAL_DATA_ROOT)
LOCAL_DATA_ROOT.mkdir(parents=True, exist_ok=True)

if DATA_SOURCE == 'drive_zip':
    assert DRIVE_DATA_ZIP.exists(), (
        f'Dataset ZIP not found: {DRIVE_DATA_ZIP}. Upload it to Drive, or edit DRIVE_DATA_ZIP.'
    )
    with zipfile.ZipFile(DRIVE_DATA_ZIP, 'r') as archive:
        archive.extractall(LOCAL_DATA_ROOT)
elif DATA_SOURCE == 'drive_folder':
    assert DRIVE_DATA_FOLDER.exists(), f'Dataset folder not found: {DRIVE_DATA_FOLDER}'
    shutil.copytree(DRIVE_DATA_FOLDER, LOCAL_DATA_ROOT, dirs_exist_ok=True)
else:
    raise ValueError("DATA_SOURCE must be 'drive_zip' or 'drive_folder'.")

if DATA_YAML_HINT is not None:
    source_yaml = LOCAL_DATA_ROOT / DATA_YAML_HINT
    assert source_yaml.exists(), f'DATA_YAML_HINT does not exist: {source_yaml}'
else:
    yaml_candidates = sorted(set(list(LOCAL_DATA_ROOT.rglob('*.yaml')) + list(LOCAL_DATA_ROOT.rglob('*.yml'))))
    yaml_candidates = [p for p in yaml_candidates if p.is_file()]
    assert len(yaml_candidates) == 1, (
        'Expected exactly one dataset YAML. Found:\n' + '\n'.join(str(p) for p in yaml_candidates) +
        '\nSet DATA_YAML_HINT in the configuration cell to choose one.'
    )
    source_yaml = yaml_candidates[0]

with source_yaml.open('r', encoding='utf-8') as f:
    dataset_cfg = yaml.safe_load(f)
assert isinstance(dataset_cfg, dict), 'Dataset YAML must be a mapping.'
assert 'train' in dataset_cfg and 'val' in dataset_cfg and 'names' in dataset_cfg, (
    'Dataset YAML must include train, val, and names.'
)

# Resolve the dataset root once, then write a YAML that always points to the current local copy.
raw_root = Path(str(dataset_cfg.get('path', source_yaml.parent)))
if not raw_root.is_absolute():
    candidate_root = (source_yaml.parent / raw_root).resolve()
    raw_root = candidate_root if candidate_root.exists() else source_yaml.parent.resolve()
dataset_cfg['path'] = str(raw_root)
LOCAL_DATA_YAML = LOCAL_DATA_ROOT / 'data_local.yaml'
with LOCAL_DATA_YAML.open('w', encoding='utf-8') as f:
    yaml.safe_dump(dataset_cfg, f, sort_keys=False, allow_unicode=True)

print('Dataset YAML selected:', source_yaml)
print('Stable local YAML:', LOCAL_DATA_YAML)
print('Resolved dataset root:', dataset_cfg['path'])
print('YAML keys:', sorted(dataset_cfg.keys()))

Dataset YAML selected: /content/lvis_fruits_yolo/data.yaml
Stable local YAML: /content/lvis_fruits_yolo/data_local.yaml
Resolved dataset root: /content/lvis_fruits_yolo
YAML keys: ['names', 'path', 'test', 'train', 'val']


## 5. Audit labels and class frequencies before training

This is a safety gate. It checks the class ID range, counts labels, detects case-only duplicate class names, and saves `class_distribution.csv` to the Drive experiment folder. Read the duplicate warning before training; do not merge classes automatically without deliberately versioning the taxonomy.

In [5]:
from pathlib import Path
from collections import Counter, defaultdict
import csv
import json

IMAGE_SUFFIXES = {'.jpg', '.jpeg', '.png', '.bmp', '.webp', '.tif', '.tiff'}

names_raw = dataset_cfg['names']
if isinstance(names_raw, dict):
    class_names = [names_raw[i] if i in names_raw else names_raw[str(i)] for i in range(len(names_raw))]
elif isinstance(names_raw, list):
    class_names = names_raw
else:
    raise TypeError('YAML names must be a list or an integer-keyed mapping.')

NC = len(class_names)
assert NC > 0, 'No classes found in YAML names.'


def resolve_dataset_path(spec):
    """Resolve current or stale YAML paths to the locally copied dataset in LOCAL_DATA_ROOT."""
    raw = Path(str(spec)).expanduser()
    candidates = [raw]

    if not raw.is_absolute():
        candidates.extend([
            Path(dataset_cfg['path']) / raw,
            LOCAL_DATA_ROOT / raw,
        ])
    else:
        # Your YAML still contains the old /content/myDrive/... location.
        # Rebuild the path from the 'images' folder inside the new local copy.
        parts = raw.parts
        if 'images' in parts:
            image_index = parts.index('images')
            image_tail = Path(*parts[image_index + 1:])
            candidates.append(LOCAL_DATA_ROOT / 'images' / image_tail)
            for local_images_dir in LOCAL_DATA_ROOT.rglob('images'):
                candidates.append(local_images_dir / image_tail)
        # This fallback handles a stale absolute .txt split list, if present.
        candidates.append(LOCAL_DATA_ROOT / raw.name)
        candidates.extend(LOCAL_DATA_ROOT.rglob(raw.name))

    seen = set()
    for candidate in candidates:
        candidate = candidate.resolve()
        if candidate in seen:
            continue
        seen.add(candidate)
        if candidate.exists():
            return candidate

    searched = '\n'.join(str(path) for path in list(seen)[:20])
    raise FileNotFoundError(
        f'Cannot resolve dataset split from: {spec}\n'
        f'First paths checked:\n{searched}\n'
        f'LOCAL_DATA_ROOT: {LOCAL_DATA_ROOT}'
    )


def split_images(spec):
    if isinstance(spec, (list, tuple)):
        paths = []
        for item in spec:
            paths.extend(split_images(item))
        return paths

    split_path = resolve_dataset_path(spec)
    if split_path.is_dir():
        return sorted(path for path in split_path.rglob('*') if path.suffix.lower() in IMAGE_SUFFIXES)
    if split_path.is_file() and split_path.suffix.lower() == '.txt':
        rows = [line.strip() for line in split_path.read_text(encoding='utf-8').splitlines() if line.strip()]
        return [resolve_dataset_path(row) for row in rows]

    raise FileNotFoundError(f'Cannot resolve images from: {spec} -> {split_path}')


def label_path_for_image(image_path):
    parts = list(image_path.parts)
    if 'images' in parts:
        image_index = parts.index('images')
        return Path(*parts[:image_index], 'labels', *parts[image_index + 1:]).with_suffix('.txt')
    return image_path.with_suffix('.txt')


def audit_split(split_name):
    image_paths = split_images(dataset_cfg[split_name])
    instance_counts = Counter()
    image_counts = Counter()
    missing_labels = 0
    invalid_rows = []

    for image_path in image_paths:
        label_path = label_path_for_image(image_path)
        if not label_path.exists():
            missing_labels += 1
            continue

        seen_here = set()
        for line_no, row in enumerate(label_path.read_text(encoding='utf-8').splitlines(), 1):
            fields = row.split()
            if not fields:
                continue
            try:
                class_id = int(float(fields[0]))
            except ValueError:
                invalid_rows.append((str(label_path), line_no, row))
                continue
            if not 0 <= class_id < NC:
                invalid_rows.append((str(label_path), line_no, row))
                continue
            instance_counts[class_id] += 1
            seen_here.add(class_id)

        for class_id in seen_here:
            image_counts[class_id] += 1

    return image_paths, instance_counts, image_counts, missing_labels, invalid_rows


train_images, train_instances, train_image_counts, train_missing, train_invalid = audit_split('train')
val_images, val_instances, val_image_counts, val_missing, val_invalid = audit_split('val')
assert not train_invalid and not val_invalid, (
    'Invalid class IDs or label rows were found. First examples:\n' +
    '\n'.join(map(str, (train_invalid + val_invalid)[:10]))
)

casefold_groups = defaultdict(list)
for idx, name in enumerate(class_names):
    casefold_groups[str(name).casefold()].append((idx, str(name)))
case_only_duplicates = {key: values for key, values in casefold_groups.items() if len(values) > 1}

print(f'Classes: {NC}')
print(f'Train images: {len(train_images):,}; labels missing: {train_missing:,}; instances: {sum(train_instances.values()):,}')
print(f'Validation images: {len(val_images):,}; labels missing: {val_missing:,}; instances: {sum(val_instances.values()):,}')
if case_only_duplicates:
    print('\nWARNING — case-only duplicate class names. Inspect before training; do not merge automatically:')
    for entries in case_only_duplicates.values():
        print(entries)
else:
    print('No case-only duplicate names detected.')

DRIVE_EXPERIMENT.mkdir(parents=True, exist_ok=True)
(DRIVE_EXPERIMENT / 'data_local.yaml').write_text(LOCAL_DATA_YAML.read_text(encoding='utf-8'), encoding='utf-8')
with (DRIVE_EXPERIMENT / 'class_distribution.csv').open('w', newline='', encoding='utf-8') as f:
    writer = csv.writer(f)
    writer.writerow(['class_id', 'class_name', 'train_images', 'train_instances', 'val_images', 'val_instances'])
    for class_id, class_name in enumerate(class_names):
        writer.writerow([
            class_id,
            class_name,
            train_image_counts[class_id],
            train_instances[class_id],
            val_image_counts[class_id],
            val_instances[class_id],
        ])

manifest = {
    'experiment': EXPERIMENT,
    'run_mode': RUN_MODE,
    'classes': NC,
    'train_images': len(train_images),
    'val_images': len(val_images),
    'train_instances': sum(train_instances.values()),
    'val_instances': sum(val_instances.values()),
    'case_only_duplicates': case_only_duplicates,
    'dataset_yaml': str(LOCAL_DATA_YAML),
}
(DRIVE_EXPERIMENT / 'dataset_audit.json').write_text(json.dumps(manifest, indent=2), encoding='utf-8')
print('\nSaved audit files to:', DRIVE_EXPERIMENT)


Classes: 63
Train images: 4,707; labels missing: 0; instances: 119,422
Validation images: 1,532; labels missing: 32; instances: 33,695

WARNING — case-only duplicate class names. Inspect before training; do not merge automatically:
[(30, 'Strawberry'), (57, 'strawberry')]
[(35, 'Tomato'), (59, 'tomato')]

Saved audit files to: /content/drive/MyDrive/FoodDetection/experiments/lvis_fruits_yolo11m_80_v1


## 6. Build the cleaned 61-class dataset — run once before training

This non-destructive processing block creates a **new versioned dataset**. It removes original class IDs **30 (`Strawberry`)** and **35 (`Tomato`)**, preserving their lower-case counterparts (original IDs 57 and 59), then remaps every retained class ID into a contiguous 0–60 range. It also removes every image with a missing or empty label file, including the 32 validation images reported by the audit.

Any image containing either removed class is discarded in full. This is intentional: retaining the image but deleting a visible object's box would turn that object into an incorrect background region and harm training. Your original dataset folder is never modified. The new dataset, mapping table, class distributions, and processing report are written to `MyDrive/FoodDetection/processed/` and copied locally for fast training.

> After this cell completes, **all later training and validation cells use `TRAIN_DATA_YAML`**, the cleaned 61-class data configuration. Do not run the old 63-class checkpoint as an exact resume: it has a different detection head. Start a new 61-class experiment from pretrained weights instead.

In [6]:
# Run this cell after the dataset-audit cell and before the training cell.
# It never changes LOCAL_DATA_ROOT or your original Google Drive dataset folder.

from pathlib import Path
from collections import Counter
import csv
import json
import shutil
from datetime import datetime, timezone
import yaml

# The user-requested taxonomy change.
DROP_CLASS_IDS = {30, 35}  # 30 = Strawberry; 35 = Tomato in the original 63-class taxonomy.
REMOVE_IMAGES_CONTAINING_DROPPED_CLASSES = True
REMOVE_IMAGES_WITH_NO_REMAINING_ANNOTATIONS = True

# Every processing change gets a new versioned destination. Do not overwrite an earlier cleaned dataset.
PROCESSED_VERSION = 'lvis_fruits_61class_drop30_35_noempty_v1'
PROCESSED_DRIVE_ROOT = DRIVE_ROOT / 'processed' / PROCESSED_VERSION
STAGING_ROOT = DRIVE_ROOT / 'processed' / f'.{PROCESSED_VERSION}.staging'
LOCAL_PROCESSED_ROOT = Path('/content/lvis_fruits_61class')

assert DROP_CLASS_IDS == {30, 35}, 'This processing block is intentionally configured only for the requested class IDs.'
assert all(0 <= class_id < NC for class_id in DROP_CLASS_IDS), 'A requested class ID is outside the original class range.'

# Read-only inventory: reveals whether another image tree in the copied Drive folder contains the missing source images.
print('Read-only image-folder inventory under LOCAL_DATA_ROOT:')
for images_directory in sorted(path for path in LOCAL_DATA_ROOT.rglob('images') if path.is_dir()):
    image_count = sum(1 for path in images_directory.rglob('*') if path.suffix.lower() in IMAGE_SUFFIXES)
    print(f'  {images_directory}: {image_count:,} images')
print(f'YAML-resolved train images: {len(train_images):,}; validation images: {len(val_images):,}')

# Process the actual splits resolved from the current YAML.
# The 6,419-image inventory includes images outside these train/validation lists (for example, a separate test split),
# so it must not be used as a required training-image count.
assert train_images, 'The YAML resolved zero training images.'
assert val_images, 'The YAML resolved zero validation images.'

assert REMOVE_IMAGES_CONTAINING_DROPPED_CLASSES, (
    'Keep this True. Retaining images that visibly contain a removed class but omitting its boxes creates false-negative labels.'
)

# New class IDs must be contiguous 0..60 for YOLO detection.
kept_old_ids = [old_id for old_id in range(NC) if old_id not in DROP_CLASS_IDS]
old_to_new = {old_id: new_id for new_id, old_id in enumerate(kept_old_ids)}
new_class_names = [class_names[old_id] for old_id in kept_old_ids]
assert len(new_class_names) == NC - len(DROP_CLASS_IDS) == 61
assert len(set(name.casefold() for name in new_class_names)) == len(new_class_names), (
    'Case-only duplicate names remain after the requested removal. Stop and inspect the taxonomy before training.'
)

# Never overwrite a finished processed version.
if PROCESSED_DRIVE_ROOT.exists():
    raise FileExistsError(
        f'Processed dataset already exists: {PROCESSED_DRIVE_ROOT}\n'
        'Change PROCESSED_VERSION to make a new version, or deliberately delete the prior processed version yourself.'
    )
if STAGING_ROOT.exists():
    shutil.rmtree(STAGING_ROOT)

for split in ('train', 'val'):
    (STAGING_ROOT / 'images' / split).mkdir(parents=True, exist_ok=True)
    (STAGING_ROOT / 'labels' / split).mkdir(parents=True, exist_ok=True)


def relative_after_images(image_path):
    """Return the path below the image root, preserving nested train/train or val/val directories."""
    parts = list(image_path.parts)
    assert 'images' in parts, f'Image path does not contain an images directory: {image_path}'
    return Path(*parts[parts.index('images') + 1:])


def parse_yolo_label(label_path):
    """Return (class_id, original row) pairs; fail loudly on malformed detection labels."""
    if not label_path.exists():
        return None
    rows = []
    for line_number, raw_row in enumerate(label_path.read_text(encoding='utf-8').splitlines(), start=1):
        fields = raw_row.split()
        if not fields:
            continue
        if len(fields) != 5:
            raise ValueError(f'Expected 5 YOLO-detection fields at {label_path}:{line_number}; found {len(fields)}: {raw_row}')
        try:
            class_id = int(fields[0])
            coordinates = [float(value) for value in fields[1:]]
        except ValueError as exc:
            raise ValueError(f'Invalid numeric YOLO label at {label_path}:{line_number}: {raw_row}') from exc
        if not 0 <= class_id < NC:
            raise ValueError(f'Class ID {class_id} outside 0..{NC - 1} at {label_path}:{line_number}')
        if not all(0.0 <= value <= 1.0 for value in coordinates):
            raise ValueError(f'YOLO coordinates must be normalized to 0..1 at {label_path}:{line_number}: {raw_row}')
        rows.append((class_id, raw_row))
    return rows


def clean_split(split_name, image_paths):
    counters = Counter()
    source_images = sorted(set(Path(path) for path in image_paths))
    output_image_paths = set()
    class_instance_counts = Counter()

    for image_path in source_images:
        label_path = label_path_for_image(image_path)
        label_rows = parse_yolo_label(label_path)

        if label_rows is None:
            counters['removed_missing_label_file'] += 1
            continue
        if not label_rows:
            counters['removed_empty_label_file'] += 1
            continue

        original_class_ids = {class_id for class_id, _ in label_rows}
        if REMOVE_IMAGES_CONTAINING_DROPPED_CLASSES and original_class_ids.intersection(DROP_CLASS_IDS):
            counters['removed_contains_requested_dropped_class'] += 1
            continue

        cleaned_rows = []
        for old_class_id, original_row in label_rows:
            if old_class_id in DROP_CLASS_IDS:
                counters['removed_annotation_for_dropped_class'] += 1
                continue
            fields = original_row.split()
            new_class_id = old_to_new[old_class_id]
            cleaned_rows.append(' '.join([str(new_class_id), *fields[1:]]))
            class_instance_counts[new_class_id] += 1

        if REMOVE_IMAGES_WITH_NO_REMAINING_ANNOTATIONS and not cleaned_rows:
            counters['removed_no_remaining_annotations'] += 1
            continue

        relative_path = relative_after_images(image_path)
        destination_image = STAGING_ROOT / 'images' / split_name / relative_path
        destination_label = STAGING_ROOT / 'labels' / split_name / relative_path.with_suffix('.txt')
        assert destination_image not in output_image_paths, f'Duplicate output image path: {destination_image}'
        output_image_paths.add(destination_image)
        destination_image.parent.mkdir(parents=True, exist_ok=True)
        destination_label.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(image_path, destination_image)
        destination_label.write_text('\n'.join(cleaned_rows) + '\n', encoding='utf-8')
        counters['kept_images'] += 1
        counters['kept_instances'] += len(cleaned_rows)

    counters['source_images'] = len(source_images)
    return counters, class_instance_counts

# Detect accidental train/validation overlap before writing the new dataset.
train_set = {Path(path).resolve() for path in train_images}
val_set = {Path(path).resolve() for path in val_images}
overlap = train_set.intersection(val_set)
assert not overlap, f'Data leakage: {len(overlap)} image(s) appear in both train and validation. First: {next(iter(overlap))}'

train_summary, train_class_counts = clean_split('train', train_images)
val_summary, val_class_counts = clean_split('val', val_images)

assert train_summary['kept_images'] > 0, 'No training images remain after processing.'
assert val_summary['kept_images'] > 0, 'No validation images remain after processing.'

# Store a Drive-portable YAML; the notebook then writes a local-path version for actual training.
processed_drive_yaml = {
    'path': str(PROCESSED_DRIVE_ROOT),
    'train': 'images/train',
    'val': 'images/val',
    'nc': len(new_class_names),
    'names': new_class_names,
}
(STAGING_ROOT / 'data.yaml').write_text(yaml.safe_dump(processed_drive_yaml, sort_keys=False, allow_unicode=True), encoding='utf-8')

with (STAGING_ROOT / 'class_id_map.csv').open('w', newline='', encoding='utf-8') as f:
    writer = csv.writer(f)
    writer.writerow(['old_class_id', 'old_class_name', 'action', 'new_class_id', 'new_class_name'])
    for old_class_id, old_class_name in enumerate(class_names):
        if old_class_id in DROP_CLASS_IDS:
            writer.writerow([old_class_id, old_class_name, 'dropped', '', ''])
        else:
            writer.writerow([old_class_id, old_class_name, 'kept', old_to_new[old_class_id], new_class_names[old_to_new[old_class_id]]])

with (STAGING_ROOT / 'class_distribution_after.csv').open('w', newline='', encoding='utf-8') as f:
    writer = csv.writer(f)
    writer.writerow(['new_class_id', 'class_name', 'train_instances', 'val_instances'])
    for new_class_id, class_name in enumerate(new_class_names):
        writer.writerow([new_class_id, class_name, train_class_counts[new_class_id], val_class_counts[new_class_id]])

processing_report = {
    'created_utc': datetime.now(timezone.utc).isoformat(),
    'source_local_dataset_root': str(LOCAL_DATA_ROOT),
    'processed_dataset_root': str(PROCESSED_DRIVE_ROOT),
    'source_class_count': NC,
    'output_class_count': len(new_class_names),
    'dropped_class_ids': sorted(DROP_CLASS_IDS),
    'dropped_class_names': [class_names[class_id] for class_id in sorted(DROP_CLASS_IDS)],
    'remove_images_containing_dropped_classes': REMOVE_IMAGES_CONTAINING_DROPPED_CLASSES,
    'remove_images_with_no_remaining_annotations': REMOVE_IMAGES_WITH_NO_REMAINING_ANNOTATIONS,
    'train': dict(train_summary),
    'val': dict(val_summary),
}
(STAGING_ROOT / 'processing_report.json').write_text(json.dumps(processing_report, indent=2), encoding='utf-8')

# Publish the completely built dataset only after all checks/copies finish.
STAGING_ROOT.replace(PROCESSED_DRIVE_ROOT)

# Copy the processed dataset to local disk; train from this writable location, not directly from Drive.
if LOCAL_PROCESSED_ROOT.exists():
    shutil.rmtree(LOCAL_PROCESSED_ROOT)
shutil.copytree(PROCESSED_DRIVE_ROOT, LOCAL_PROCESSED_ROOT)

local_training_cfg = yaml.safe_load((LOCAL_PROCESSED_ROOT / 'data.yaml').read_text(encoding='utf-8'))
local_training_cfg['path'] = str(LOCAL_PROCESSED_ROOT)
TRAIN_DATA_YAML = LOCAL_PROCESSED_ROOT / 'data_local.yaml'
TRAIN_DATA_YAML.write_text(yaml.safe_dump(local_training_cfg, sort_keys=False, allow_unicode=True), encoding='utf-8')

print('Processing complete.')
print('New class count:', len(new_class_names))
print('Dropped:', [(class_id, class_names[class_id]) for class_id in sorted(DROP_CLASS_IDS)])
print('Train:', dict(train_summary))
print('Validation:', dict(val_summary))
print('Persistent processed dataset:', PROCESSED_DRIVE_ROOT)
print('Training YAML (use this in model.train):', TRAIN_DATA_YAML)


Read-only image-folder inventory under LOCAL_DATA_ROOT:
  /content/lvis_fruits_yolo/images: 6,419 images
YAML-resolved train images: 4,707; validation images: 1,532
Processing complete.
New class count: 61
Dropped: [(30, 'Strawberry'), (35, 'Tomato')]
Train: {'kept_images': 4695, 'kept_instances': 119375, 'removed_contains_requested_dropped_class': 12, 'source_images': 4707}
Validation: {'kept_images': 1492, 'kept_instances': 33683, 'removed_contains_requested_dropped_class': 8, 'removed_missing_label_file': 32, 'source_images': 1532}
Persistent processed dataset: /content/drive/MyDrive/FoodDetection/processed/lvis_fruits_61class_drop30_35_noempty_v1
Training YAML (use this in model.train): /content/lvis_fruits_61class/data_local.yaml


## 6. Start, resume, or fine-tune training with Drive-backed checkpoints

The callback runs after each model checkpoint save. It atomically copies the current full `last.pt` and `best.pt` into Drive, then copies configuration and metrics files. An extra immutable epoch checkpoint is retained every five epochs. This means a sudden disconnect costs no more than the incomplete epoch.

The first `fresh` run records `epochs=80` inside `last.pt`. For an interruption, set `RUN_MODE = 'resume_interrupted'`, keep the same `EXPERIMENT`, re-run all cells above, and execute this cell. Do not change optimizer, LR, model, dataset, or epoch target in an exact-resume run; those settings come from the checkpoint.

In [7]:
import platform
import shutil
from datetime import datetime, timezone
from ultralytics import YOLO

DRIVE_WEIGHTS = DRIVE_EXPERIMENT / 'weights'
DRIVE_WEIGHTS.mkdir(parents=True, exist_ok=True)
LOCAL_RUNS_ROOT.mkdir(parents=True, exist_ok=True)

# Persist this notebook's intentional settings before training starts.
config_snapshot = {
    'created_utc': datetime.now(timezone.utc).isoformat(),
    'experiment': EXPERIMENT,
    'run_mode': RUN_MODE,
    'fresh_model': FRESH_MODEL,
    'target_epochs': TARGET_EPOCHS,
    'finetune_epochs': FINETUNE_EPOCHS,
    'imgsz': IMGSZ,
    'batch': BATCH,
    'workers': WORKERS,
    'cache': CACHE,
    'optimizer': OPTIMIZER,
    'lr0': LR0,
    'momentum': MOMENTUM,
    'weight_decay': WEIGHT_DECAY,
    'warmup_epochs': WARMUP_EPOCHS,
    'lrf': LRF,
    'box': BOX_LOSS_GAIN,
    'cls': CLS_LOSS_GAIN,
    'dfl': DFL_LOSS_GAIN,
    'seed': SEED,
    'python': platform.python_version(),
    'torch': torch.__version__,
    'ultralytics': ultralytics.__version__,
    'gpu': torch.cuda.get_device_name(0),
    'dataset_yaml': str(TRAIN_DATA_YAML),
}
(DRIVE_EXPERIMENT / 'training_config.json').write_text(json.dumps(config_snapshot, indent=2), encoding='utf-8')

# Copy safely so an interrupted Drive write does not destroy the previously valid checkpoint.
def atomic_copy(source, destination):
    source = Path(source)
    destination = Path(destination)
    if not source.exists():
        return False
    destination.parent.mkdir(parents=True, exist_ok=True)
    temporary = destination.with_suffix(destination.suffix + '.partial')
    shutil.copy2(source, temporary)
    temporary.replace(destination)
    return True

def persist_to_drive(trainer):
    # Called after a checkpoint save; preserves exact-resume state and experiment metadata.
    try:
        local_run = Path(trainer.save_dir)
        copied = []
        for weight in (local_run / 'weights').glob('*.pt'):
            # Keep last/best each epoch; keep epoch-number files according to SAVE_PERIOD.
            if atomic_copy(weight, DRIVE_WEIGHTS / weight.name):
                copied.append(weight.name)
        for filename in ('args.yaml', 'results.csv', 'results.png', 'labels.jpg', 'PR_curve.png', 'F1_curve.png', 'P_curve.png', 'R_curve.png'):
            atomic_copy(local_run / filename, DRIVE_EXPERIMENT / filename)
        status = {
            'saved_utc': datetime.now(timezone.utc).isoformat(),
            'completed_epoch_index': int(getattr(trainer, 'epoch', -1)),
            'checkpoint_files': sorted(copied),
            'local_run': str(local_run),
        }
        (DRIVE_EXPERIMENT / 'checkpoint_status.json').write_text(json.dumps(status, indent=2), encoding='utf-8')
        print(f'\n[Drive backup complete after epoch {status["completed_epoch_index"] + 1}: {", ".join(copied)}]')
    except Exception as exc:
        # Do not crash the active training job; instead make the loss of persistence obvious in the notebook output.
        print(f'\nWARNING: checkpoint backup to Drive failed: {type(exc).__name__}: {exc}')

if RUN_MODE == 'fresh':
    # Audit/config files are expected already; only existing checkpoints imply an overwrite risk.
    if any(DRIVE_WEIGHTS.glob('*.pt')):
        raise FileExistsError(
            f'Experiment already has checkpoints in: {DRIVE_WEIGHTS}. '
            'Choose a new EXPERIMENT name for a fresh run, or choose a resume/extend mode.'
        )
    DRIVE_EXPERIMENT.mkdir(parents=True, exist_ok=True)
    model = YOLO(FRESH_MODEL)
    model.add_callback('on_model_save', persist_to_drive)
    results = model.train(
        data=str(TRAIN_DATA_YAML),
        epochs=TARGET_EPOCHS,
        imgsz=IMGSZ,
        batch=BATCH,
        device=DEVICE,
        workers=WORKERS,
        cache=CACHE,
        optimizer=OPTIMIZER,
        lr0=LR0,
        momentum=MOMENTUM,
        weight_decay=WEIGHT_DECAY,
        warmup_epochs=WARMUP_EPOCHS,
        lrf=LRF,
        cos_lr=True,
        box=BOX_LOSS_GAIN,
        cls=CLS_LOSS_GAIN,
        dfl=DFL_LOSS_GAIN,
        patience=PATIENCE,
        close_mosaic=10,
        amp=True,
        deterministic=True,
        seed=SEED,
        save=True,
        save_period=SAVE_PERIOD,
        project=str(LOCAL_RUNS_ROOT),
        name=EXPERIMENT,
        exist_ok=False,
        plots=True,
        val=True,
        save_json=False,
        verbose=True,
    )
elif RUN_MODE == 'resume_interrupted':
    assert PREVIOUS_CHECKPOINT.exists(), f'No resume checkpoint: {PREVIOUS_CHECKPOINT}'
    model = YOLO(str(PREVIOUS_CHECKPOINT))
    model.add_callback('on_model_save', persist_to_drive)
    # Exact resume: state and arguments are restored from the checkpoint.
    results = model.train(resume=True)
elif RUN_MODE == 'extend_completed':
    assert PREVIOUS_CHECKPOINT.exists(), f'No starting weights: {PREVIOUS_CHECKPOINT}'
    model = YOLO(str(PREVIOUS_CHECKPOINT))
    model.add_callback('on_model_save', persist_to_drive)
    # This is a new fine-tuning stage, deliberately not resume=True.
    results = model.train(
        data=str(TRAIN_DATA_YAML),
        epochs=FINETUNE_EPOCHS,
        imgsz=IMGSZ,
        batch=BATCH,
        device=DEVICE,
        workers=WORKERS,
        cache=CACHE,
        optimizer=OPTIMIZER,
        lr0=LR0 * 0.5,
        momentum=MOMENTUM,
        weight_decay=WEIGHT_DECAY,
        warmup_epochs=1.0,
        lrf=LRF,
        cos_lr=True,
        box=BOX_LOSS_GAIN,
        cls=CLS_LOSS_GAIN,
        dfl=DFL_LOSS_GAIN,
        patience=PATIENCE,
        close_mosaic=10,
        amp=True,
        deterministic=True,
        seed=SEED,
        save=True,
        save_period=SAVE_PERIOD,
        project=str(LOCAL_RUNS_ROOT),
        name=EXPERIMENT + '_finetune',
        exist_ok=False,
        plots=True,
        val=True,
        save_json=False,
        verbose=True,
    )
else:
    raise ValueError("RUN_MODE must be 'fresh', 'resume_interrupted', or 'extend_completed'.")

# One final copy attempts to capture plots and the final best checkpoint.
if 'trainer' in globals():
    persist_to_drive(trainer)
print('\nTraining call completed. Persistent files are in:', DRIVE_EXPERIMENT)

New https://pypi.org/project/ultralytics/8.4.127 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.3.32 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: task=detect, mode=train, model=/content/drive/MyDrive/lvis_epoch45_recovery/lvis_fruits_yolo11m_80_v1/weights/last.pt, data=/content/lvis_fruits_61class/data_local.yaml, epochs=80, time=None, patience=20, batch=13, imgsz=640, save=True, save_period=5, cache=disk, device=0, workers=2, project=/content/yolo_runs, name=lvis_fruits_yolo11m_80_v1, exist_ok=False, pretrained=True, optimizer=AdamW, verbose=True, seed=0, deterministic=True, single_cls=False, rect=False, cos_lr=True, close_mosaic=10, resume=/content/drive/MyDrive/lvis_epoch45_recovery/lvis_fruits_yolo11m_80_v1/weights/last.pt, amp=True, fraction=1.0, profile=False, freeze=None, multi_scale=False, overlap_mask=True, mask_ratio=4, dropout=0.0, val=True, split=val, save_json=False, save_hybrid=False, conf=None, iou=0.7, max_det=300, 

100%|██████████| 755k/755k [00:00<00:00, 154MB/s]


TensorBoard: Start with 'tensorboard --logdir /content/yolo_runs/lvis_fruits_yolo11m_80_v1', view at http://localhost:6006/

                   from  n    params  module                                       arguments                     
  0                  -1  1      1856  ultralytics.nn.modules.conv.Conv             [3, 64, 3, 2]                 
  1                  -1  1     73984  ultralytics.nn.modules.conv.Conv             [64, 128, 3, 2]               
  2                  -1  1    111872  ultralytics.nn.modules.block.C3k2            [128, 256, 1, True, 0.25]     
  3                  -1  1    590336  ultralytics.nn.modules.conv.Conv             [256, 256, 3, 2]              
  4                  -1  1    444928  ultralytics.nn.modules.block.C3k2            [256, 512, 1, True, 0.25]     
  5                  -1  1   2360320  ultralytics.nn.modules.conv.Conv             [512, 512, 3, 2]              
  6                  -1  1   1380352  ultralytics.nn.modules.block.C3k2      

100%|██████████| 5.35M/5.35M [00:00<00:00, 399MB/s]


AMP: checks passed ✅


...

                   all       1492      33683       0.35      0.249      0.234      0.169

[Drive backup complete after epoch 79: epoch60.pt, epoch70.pt, best.pt, epoch75.pt, epoch55.pt, last.pt, epoch65.pt, epoch50.pt]

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size


      80/80      11.8G      0.894     0.6203     0.9509          5        640: 100%|██████████| 362/362 [02:56<00:00,  2.05it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 58/58 [00:21<00:00,  2.65it/s]


                   all       1492      33683      0.352      0.247      0.234       0.17
EarlyStopping: Training stopped early as no improvement observed in last 20 epochs. Best results observed at epoch 60, best model saved as best.pt.
To update EarlyStopping(patience=20) pass a new patience value, i.e. `patience=300` or use `patience=0` to disable EarlyStopping.

[Drive backup complete after epoch 80: epoch60.pt, epoch70.pt, best.pt, epoch75.pt, epoch55.pt, last.pt, epoch65.pt, epoch50.pt]

31 epochs completed in 1.837 hours.
Optimizer stripped from /content/yolo_runs/lvis_fruits_yolo11m_80_v1/weights/last.pt, 40.6MB
Optimizer stripped from /content/yolo_runs/lvis_fruits_yolo11m_80_v1/weights/best.pt, 40.6MB

Validating /content/yolo_runs/lvis_fruits_yolo11m_80_v1/weights/best.pt...
Ultralytics 8.3.32 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
YOLO11m summary (fused): 303 layers, 20,077,063 parameters, 0 gradients, 67.9 GFLOPs


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 58/58 [00:33<00:00,  1.76it/s]


                   all       1492      33683      0.364      0.265      0.243      0.177
                almond          7        132      0.401     0.0909      0.143      0.109
                 apple        184       3529      0.578      0.535      0.571      0.456
               apricot          3         12      0.423      0.333       0.27      0.256
             artichoke         11        102       0.39      0.225      0.201      0.123
             asparagus         13        108      0.232      0.306      0.251       0.13
               avocado         22        138      0.561      0.225      0.268      0.214
                banana        312       8060      0.512       0.67      0.615      0.408
        bean curd/tofu          5         36     0.0283     0.0278      0.016     0.0126
  bell pepper/capsicum         57        627      0.308      0.396      0.291      0.181
            blackberry          4         34      0.418      0.324      0.349      0.273
             blueberr

In [8]:
from pathlib import Path
from google.colab import files
import zipfile

RUN_DIR = Path('/content/yolo_runs/lvis_fruits_yolo11m_80_v1')
BACKUP_ZIP = Path('/content/lvis_trainning_results.zip')

required = [
    RUN_DIR / 'weights' / 'last.pt',  # essential: exact resume
    RUN_DIR / 'weights' / 'best.pt',
    RUN_DIR / 'results.csv',
    RUN_DIR / 'args.yaml',
]

missing = [str(path) for path in required if not path.exists()]
assert not missing, f'Missing files: {missing}. Check RUN_DIR against the training log.'

with zipfile.ZipFile(BACKUP_ZIP, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
    for path in required:
        archive.write(path, path.relative_to(RUN_DIR.parent))

print('Created:', BACKUP_ZIP, f'({BACKUP_ZIP.stat().st_size / 1024 / 1024:.1f} MB)')
files.download(str(BACKUP_ZIP))


Created: /content/lvis_trainning_results.zip (71.4 MB)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 7. Validate the durable best checkpoint and inspect learning curves

Validation is run from the Drive-backed best checkpoint, not from an in-memory Python model. That makes the final result repeatable after a later Colab session. `mAP50-95`, per-class AP, precision, and recall matter more than training loss alone.

In [10]:
from IPython.display import Image, display

BEST_CHECKPOINT = DRIVE_WEIGHTS / 'best.pt'
if not BEST_CHECKPOINT.exists():
    # In a fresh run, the local folder is a fallback before the first Drive callback completes.
    fallback_best = LOCAL_EXPERIMENT / 'weights' / 'best.pt'
    assert fallback_best.exists(), 'best.pt not found. Confirm that training completed or that Drive backup succeeded.'
    BEST_CHECKPOINT = fallback_best

# Ultralytics validation requires a strictly positive integer batch size.
try:
    VAL_BATCH = int(BATCH)
except (TypeError, ValueError):
    VAL_BATCH = 1

if VAL_BATCH <= 0:
    VAL_BATCH = 1

best_model = YOLO(str(BEST_CHECKPOINT))
metrics = best_model.val(
    data=str(TRAIN_DATA_YAML),
    imgsz=IMGSZ,
    batch=VAL_BATCH,
    device=DEVICE,
    workers=WORKERS,
    cache=CACHE,
    split='val',
    plots=True,
    save_json=False,
)
print('Validated checkpoint:', BEST_CHECKPOINT)
print('Validation batch size:', VAL_BATCH)
print('Validation metrics object:', metrics)

for plot_name in ('results.png', 'PR_curve.png', 'F1_curve.png', 'confusion_matrix_normalized.png'):
    candidate = DRIVE_EXPERIMENT / plot_name
    if candidate.exists():
        display(Image(filename=str(candidate)))


Ultralytics 8.3.32 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
YOLO11m summary (fused): 303 layers, 20,077,063 parameters, 0 gradients, 67.9 GFLOPs


val: Scanning /content/lvis_fruits_61class/labels/val/val/val.cache... 1492 images, 0 backgrounds, 0 corrupt: 100%|██████████| 1492/1492 [00:00<?, ?it/s]
val: Caching images (1.2GB Disk): 100%|██████████| 1492/1492 [00:00<00:00, 41972.30it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 1492/1492 [00:51<00:00, 29.12it/s]


                   all       1492      33683      0.372      0.261      0.243      0.177
                almond          7        132      0.431     0.0833      0.158      0.121
                 apple        184       3529      0.591      0.532      0.571      0.457
               apricot          3         12       0.44      0.333      0.271      0.256
             artichoke         11        102      0.402      0.217      0.188      0.112
             asparagus         13        108      0.246      0.315      0.258      0.132
               avocado         22        138      0.629      0.225      0.273      0.216
                banana        312       8060      0.521      0.663      0.614      0.408
        bean curd/tofu          5         36     0.0298     0.0278     0.0167     0.0133
  bell pepper/capsicum         57        627      0.316      0.391      0.292      0.181
            blackberry          4         34      0.452      0.324      0.364      0.269
             blueberr

## 8. Export the durable experiment package

The ZIP contains weights, metrics, configuration snapshots, the local YAML, label-distribution report, and plots. It deliberately excludes the raw training dataset, which remains in `source/` as the immutable input archive.

In [12]:
import shutil

# Final small-file synchronization in case a plot was produced after the last checkpoint callback.
for filename in ('args.yaml', 'results.csv', 'results.png', 'labels.jpg', 'PR_curve.png', 'F1_curve.png', 'P_curve.png', 'R_curve.png'):
    local_candidate = LOCAL_EXPERIMENT / filename
    if local_candidate.exists():
        atomic_copy(local_candidate, DRIVE_EXPERIMENT / filename)

export_stem = DRIVE_EXPORTS / f'{EXPERIMENT}_artifacts'
archive_path = shutil.make_archive(str(export_stem), 'zip', root_dir=DRIVE_EXPERIMENT)
print('Experiment archive:', archive_path)
print('Drive experiment folder:', DRIVE_EXPERIMENT)
print('Key recovery checkpoint:', DRIVE_WEIGHTS / 'last.pt')

Experiment archive: /content/drive/MyDrive/FoodDetection/exports/lvis_fruits_yolo11m_80_v1_artifacts.zip
Drive experiment folder: /content/drive/MyDrive/FoodDetection/experiments/lvis_fruits_yolo11m_80_v1
Key recovery checkpoint: /content/drive/MyDrive/FoodDetection/experiments/lvis_fruits_yolo11m_80_v1/weights/last.pt


## Recovery checklist

If Colab expires or disconnects, open this same notebook in a new session. Run sections 1–5 to mount Drive, reinstall the pinned package, and recreate `/content/lvis_fruits_yolo` from the immutable source ZIP. In section 3, retain the same `EXPERIMENT`, set `RUN_MODE = 'resume_interrupted'`, and make `PREVIOUS_CHECKPOINT` point to `experiments/<EXPERIMENT>/weights/last.pt`. Then run section 6.

Do **not** choose `fresh` with an existing experiment folder. Do **not** use `resume_interrupted` from only `best.pt` after a normal completed run. For that situation use `extend_completed`, choose a new experiment name, and record it as a separate fine-tuning stage.

### Parameter rationale summary

| Parameter | Selected value | Reason |
|---|---:|---|
| `FRESH_MODEL` | `yolo11m.pt` | Medium model with a more economical compute profile than the original YOLOv9c; practical for a variable free GPU. |
| `TARGET_EPOCHS` | 80 | The original 30-epoch curves were still improving, so 30 was too early to judge convergence. |
| `imgsz` | 640 | Preserves a controlled comparison with the original run. |
| `batch` | -1 | Uses the safe batch for the particular GPU allocated today. |
| `optimizer` | AdamW | Makes explicit the optimizer automatically selected in the original log. |
| `lr0` | 0.000149 | Reuses the learning rate actually used by the original auto-optimizer. |
| `box`, `cls`, `dfl` | 7.5, 0.5, 1.5 | Keeps the standard detector loss balance; no evidence yet supports changing it. |
| `cache` | disk | Uses fast, writable runtime storage without consuming Drive I/O for each batch. |
| `save_period` | 5 | Retains periodic historical checkpoints while `last.pt` is synchronised to Drive every completed epoch. |
| `patience` | 20 | Stops unnecessary epochs when validation fitness genuinely plateaus. |

**References:** [Ultralytics training and resume](https://docs.ultralytics.com/modes/train/), [Ultralytics callbacks](https://docs.ultralytics.com/usage/callbacks/), [Ultralytics YOLO11 model table](https://docs.ultralytics.com/models/yolo11/), [Google Colab FAQ](https://research.google.com/colaboratory/faq.html), and [LVIS project](https://www.lvisdataset.org/).

In [13]:
from google.colab import drive
from pathlib import Path
import shutil

# Mount Google Drive
drive.mount('/content/drive')

# Destination folder in Google Drive
destination = Path('/content/drive/MyDrive/colab_backup')
destination.mkdir(parents=True, exist_ok=True)

# Files and folders to copy
sources = [
    Path('/content/yolo_runs'),
    Path('/content/lvis_trainning_results.zip'),
    Path('/content/runs'),
    Path('/content/lvis_fruits_61class'),
    Path('/content/drive/MyDrive/FoodDetection'),
]

for source in sources:
    target = destination / source.name

    if not source.exists():
        print(f'NOT FOUND: {source}')
        continue

    try:
        if source.is_dir():
            # Copies the folder and merges with an existing destination folder if needed.
            shutil.copytree(source, target, dirs_exist_ok=True)
        else:
            target.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(source, target)

        print(f'COPIED: {source}  ->  {target}')

    except Exception as error:
        print(f'ERROR copying {source}: {error}')

print('\nBackup completed.')
print(f'Google Drive destination: {destination}')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
COPIED: /content/yolo_runs  ->  /content/drive/MyDrive/colab_backup/yolo_runs
COPIED: /content/lvis_trainning_results.zip  ->  /content/drive/MyDrive/colab_backup/lvis_trainning_results.zip
COPIED: /content/runs  ->  /content/drive/MyDrive/colab_backup/runs
COPIED: /content/lvis_fruits_61class  ->  /content/drive/MyDrive/colab_backup/lvis_fruits_61class
COPIED: /content/drive/MyDrive/FoodDetection  ->  /content/drive/MyDrive/colab_backup/FoodDetection

Backup completed.
Google Drive destination: /content/drive/MyDrive/colab_backup
